# Sensitivities and inference

How a result depends on the rate data, how uncertain it is, and how to infer $T_e$ and $n_e$ from line ratios. The methods are in [Sensitivities and inference](/OpenCR/guides/inference/).

In [ ]:
import numpy as np
import plasma_plots

import opencr
from opencr.inference import (
    Observation,
    fit,
    forward_model,
    posterior_grid,
    propagate,
    sensitivities,
)

model = opencr.load_model("hydrogen/atomic/basic", n_max=10)
conditions = opencr.Conditions(5.0, 1e18, densities={"H(n=1)": 1e18, "H+": 1e18})
balmer = "ratio:emissivity:A:3-2/emissivity:A:4-2"
ionization = "effective_rate:H(n=1)>H+"

## Sensitivities

The logarithmic sensitivities of the CR ionization coefficient and the Balmer α/β ratio to the reactions that matter most:

In [ ]:
s = sensitivities(model, conditions, [ionization, balmer])
for output in s.output.values:
    row = s.sel(output=output)
    top = np.argsort(-np.abs(row.values))[:4]
    print(output)
    for i in top:
        print(f"  {str(row.reaction.values[i]):10s} {row.values[i]:+.3f}")

## Uncertainty from the data

With the uncertainties in the reactions' provenance (20% for the excitation and ionization fits), linearly and by Monte Carlo:

In [ ]:
linear = propagate(model, conditions, [ionization, balmer])
carlo = propagate(
    model, conditions, [ionization, balmer], method="monte-carlo", samples=200
)
for i, output in enumerate(linear.outputs):
    print(
        f"{output}: {100 * linear.relative[i]:.1f}% (linear), {100 * carlo.relative[i]:.1f}% (Monte Carlo)"
    )
print(linear.dominant(ionization, 3))

## $T_e$ and $n_e$ from line ratios

Synthetic measurements at 3 eV and 3e18 m⁻³ with 5% uncertainty: two Balmer ratios and Lyman α to Balmer α. The grid posterior shows how well the ratios constrain the two parameters:

In [ ]:
outputs = [
    balmer,
    "ratio:emissivity:A:3-2/emissivity:A:5-2",
    "ratio:emissivity:A:2-1/emissivity:A:3-2",
]
forward = forward_model(
    model, conditions, ["electron_temperature", "electron_density"], outputs
)
measured = forward(np.array([3.0, 3e18]))
observations = [
    Observation(n, float(v), 0.05) for n, v in zip(outputs, measured, strict=True)
]
bounds = {"electron_temperature": (1.0, 20.0), "electron_density": (1e17, 1e20)}
best = fit(forward, observations, bounds)
print(best.parameters, best.log_std)
grid = posterior_grid(forward, observations, bounds, points=21)
with plasma_plots.figure(1, 1) as fig:
    fig[0].contourf(
        grid.electron_temperature, grid.electron_density, grid.posterior.T, levels=12
    )
    fig[0].set_xscale("log")
    fig[0].set_yscale("log")
    fig[0].plot([3.0], [3e18], "w+", markersize=12)
    fig[0].set_xlabel("$T_e$ [eV]")
    fig[0].set_ylabel("$n_e$ [m$^{-3}$]")